In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
SILVER_REGION_TABLE = "us_grid_energy_pipeline_databricks.usgrid.silver_region_data"
DIM_METRIC_TABLE    = "us_grid_energy_pipeline_databricks.usgrid.dim_metric_type"
DIM_REGION_TABLE    = "us_grid_energy_pipeline_databricks.usgrid.dim_region"

GOLD_PATH  = "abfss://gold@usgridenergypipeline.dfs.core.windows.net/gold_region_metrics"
GOLD_TABLE = "us_grid_energy_pipeline_databricks.usgrid.gold_region_metrics"

In [0]:
# Create Gold Table gold_region_metrics

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        region_code          STRING,
        region_name          STRING,
        period               TIMESTAMP,
        demand_mw            FLOAT,
        forecast_mw          FLOAT,
        net_generation_mw    FLOAT,
        total_interchange_mw FLOAT,
        forecast_error_mw    FLOAT
    )
    USING DELTA
    LOCATION '{GOLD_PATH}'
    PARTITIONED BY (region_code)
""")

DataFrame[]

In [0]:
df_silver = spark.read.table(SILVER_REGION_TABLE)
df_dim_metric = spark.read.table(DIM_METRIC_TABLE)
df_dim_region = spark.read.table(DIM_REGION_TABLE)

In [0]:
df_pivoted = (
    df_silver
    .groupBy("region_code", "period")
    .pivot("metric_type_code", ["D", "DF", "NG", "TI"])
    .agg(F.first("value_in_megawatts"))
    .withColumnRenamed("D",  "demand_mw")
    .withColumnRenamed("DF", "forecast_mw")
    .withColumnRenamed("NG", "net_generation_mw")
    .withColumnRenamed("TI", "total_interchange_mw")
)

In [0]:
df_joined = (
    df_pivoted
    .join(df_dim_region, on="region_code", how="left")
)

In [0]:
df_metrics = (
    df_joined
    .withColumn(
        "forecast_error_mw",
        F.col("demand_mw") - F.col("forecast_mw")
    )
)

In [0]:
df_gold = df_metrics.select(
    "region_code",
    "region_name",
    "period",
    "demand_mw",
    "forecast_mw",
    "net_generation_mw",
    "total_interchange_mw",
    "forecast_error_mw"
)

In [0]:
df_gold.createOrReplaceTempView("gold_region_updates")

spark.sql(f"""
    MERGE INTO {GOLD_TABLE} AS target
    USING gold_region_updates AS source
    ON target.region_code = source.region_code
    AND target.period = source.period
    WHEN MATCHED AND (
        target.demand_mw != source.demand_mw OR
        target.forecast_mw != source.forecast_mw OR
        target.net_generation_mw != source.net_generation_mw OR
        target.total_interchange_mw != source.total_interchange_mw
    )
        THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Verification
spark.sql(f"SELECT * FROM {GOLD_TABLE} LIMIT 10").display()

region_code,region_name,period,demand_mw,forecast_mw,net_generation_mw,total_interchange_mw,forecast_error_mw
NYIS,New York Independent System Operator,2026-03-01T07:00:00Z,15420.0,15375.0,12433.0,-2987.0,45.0
NYIS,New York Independent System Operator,2025-03-14T07:00:00Z,14171.0,0.0,12012.0,-2159.0,14171.0
NYIS,New York Independent System Operator,2025-03-04T20:00:00Z,17155.0,16853.0,14964.0,-2191.0,302.0
NYIS,New York Independent System Operator,2025-03-28T13:00:00Z,16552.0,16435.0,13800.0,-2752.0,117.0
NYIS,New York Independent System Operator,2026-01-12T20:00:00Z,18649.0,18158.0,16669.0,-1980.0,491.0
NYIS,New York Independent System Operator,2026-01-17T03:00:00Z,19809.0,18859.0,17505.0,-2304.0,950.0
NYIS,New York Independent System Operator,2025-02-13T13:00:00Z,19070.0,18322.0,16803.0,-2267.0,748.0
NYIS,New York Independent System Operator,2025-02-18T03:00:00Z,20250.0,19715.0,16799.0,-3451.0,535.0
NYIS,New York Independent System Operator,2025-02-07T12:00:00Z,18150.0,17464.0,16304.0,-1846.0,686.0
NYIS,New York Independent System Operator,2026-02-07T11:00:00Z,18159.0,17412.0,14365.0,-3794.0,747.0


In [0]:
# Row Count check
spark.sql(f"""
    SELECT region_code, region_name, COUNT(*) as row_count
    FROM {GOLD_TABLE}
    GROUP BY region_code, region_name
    ORDER BY region_code
""").display()

region_code,region_name,row_count
CISO,California Independent System Operator,4104
ERCO,"Electric Reliability Council of Texas, Inc.",4128
ISNE,ISO New England,4104
MISO,"Midcontinent Independent System Operator, Inc.",4104
NYIS,New York Independent System Operator,4104
PJM,"PJM Interconnection, LLC",4080
SWPP,Southwest Power Pool,4104
